<a href="https://colab.research.google.com/github/visista19/30-days-challenge/blob/main/Q%26Asystem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentence-transformers datasets pandas scikit-learn

In [ ]:
import pandas as pd
from datasets import load_dataset

# 1. Stream the real SQuAD dataset from Hugging Face
squad_stream = load_dataset("rajpurkar/squad", split="train", streaming=True)
squad_head = list(squad_stream.take(400))

# 2. Extract real questions and their matching answer contexts
questions_list = [row['question'] for row in squad_head]
contexts_list = [row['context'] for row in squad_head]

# 3. Build your real Q&A DataFrame
real_qa_database = pd.DataFrame({
    'input_text': questions_list,   # The real questions we embed and search against
    'output_text': contexts_list    # The real text paragraphs that contain the answers!
})

print(f"Successfully loaded {len(real_qa_database)} rows from SQuAD!")
print("\nSample Data Entry:")
print(f"Question: {real_qa_database.input_text[159]}")
print(f"Answer Context: {real_qa_database.output_text[0][:150]}...")

Successfully loaded 400 rows from SQuAD!

Sample Data Entry:
Question: Notre Dame has a center in Beijing, what is it referred to as?
Answer Context: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front o...


In [ ]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
database_questions = real_qa_database['input_text'].tolist()
question_embeddings = embedding_model.encode(database_questions)
real_qa_database['embeddings'] = list(question_embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
user_query = "Notre Dame has a center in Beijing, what is it referred to as"
query_embedding = embedding_model.encode([user_query])
cos_sim_array = cosine_similarity(query_embedding, list(real_qa_database['embeddings']))
index_doc_cosine = np.argmax(cos_sim_array)
confidence_score = cos_sim_array[0][index_doc_cosine]

# 5. Output the results
print(f"Your Query: {user_query}")
print(f"Search Confidence Score: {confidence_score:.4f}\n")
print("Closest Matched Question in Database:")
print(real_qa_database.input_text[index_doc_cosine])
print("\nReturned Answer Context:")
print(real_qa_database.output_text[index_doc_cosine])


Your Query: Notre Dame has a center in Beijing, what is it referred to as
Search Confidence Score: 0.9556

Closest Matched Question in Database:
Notre Dame has a center in Beijing, what is it referred to as?

Returned Answer Context:
The university owns several centers around the world used for international studies and research, conferences abroad, and alumni support. The university has had a presence in London, England, since 1968. Since 1998, its London center has been based in the former United University Club at 1 Suffolk Street in Trafalgar Square. The center enables the Colleges of Arts & Letters, Business Administration, Science, Engineering and the Law School to develop their own programs in London, as well as hosting conferences and symposia. Other Global Gateways are located in Beijing, Chicago, Dublin, Jerusalem and Rome.
